# 06 Resume Positioning

Stage 6 generates the resume's top-level positioning from the evidence-backed capability review.

This stage creates the Summary and Core Expertise sections. These sections translate demonstrated capabilities into concise resume language while preserving traceability back to selected evidence.

Inputs:
- `artifacts/capability_review.json`
- `artifacts/selected_evidence.json`
- `artifacts/target_archetype.json`

Output:
- `artifacts/resume_positioning.json`

This is resume language generation, but it is not full resume assembly. The model may choose phrasing, but it may not introduce unsupported claims.

### Output shape
```json
{
  "target_title": "Principal AI Architect / AI Platform Engineering Lead",
  "summary": {
    "paragraph": "string",
    "supporting_capabilities": ["string"],
    "supporting_evidence_ids": ["string"],
    "cautions": ["string"]
  },
  "core_expertise": {
    "items": [
      {
        "label": "Enterprise AI Architecture",
        "supporting_capabilities": ["string"],
        "supporting_evidence_ids": ["string"]
      }
    ]
  },
  "positioning_notes": {
    "lead_story": "string",
    "avoid_overstating": ["string"],
    "experience_section_guidance": ["string"]
  }
}
```

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import json
import pandas as pd

from src.config import ARTIFACT_DIR
from src.helpers import load_json, save_json
from src.capabilities import call_json_model

load_dotenv()

MODEL = ChatOpenAI(
    model="gpt-4.1",
    temperature=0,
)

## 6A. Load Inputs

In [2]:
capability_review = load_json(ARTIFACT_DIR / "capability_review.json")
selected_evidence = load_json(ARTIFACT_DIR / "selected_evidence.json")
target_archetype = load_json(ARTIFACT_DIR / "target_archetype.json")

capability_review.keys(), selected_evidence.keys(), target_archetype.keys()

(dict_keys(['capability_review_summary', 'capabilities']),
 dict_keys(['selection_summary', 'selected_evidence', 'excluded_evidence']),
 dict_keys(['title', 'archetype_summary', 'market_basis', 'hypothesis_assessment', 'core_market_themes', 'target_capabilities', 'target_problem_spaces', 'target_technology_areas', 'seniority_expectations', 'success_patterns', 'resume_implications', 'archetype_narrative']))

In [3]:
capability_review["capability_review_summary"]

{'target_archetype': 'Principal AI Architect / AI Platform Engineering Lead',
 'source_artifact': 'selected_evidence.json',
 'capability_count': 12,
 'method': 'Capabilities inferred from selected evidence and target archetype.'}

## 6B. Prepare Inputs for Positioning

In [4]:
POSITIONING_EVIDENCE_TIERS = {"must_include", "strong_include", "optional"}

positioning_evidence = [
    item
    for item in selected_evidence["selected_evidence"]
    if item["selection_tier"] in POSITIONING_EVIDENCE_TIERS
]

len(positioning_evidence)

23

In [5]:
def compact_capabilities_for_positioning(capability_review):
    return [
        {
            "capability": cap.get("capability"),
            "original_capability": cap.get("original_capability", cap.get("capability")),
            "strength": cap.get("strength"),
            "target_relevance": cap.get("target_relevance"),
            "resume_use": cap.get("resume_use"),
            "evidence_ids": cap.get("evidence_ids", []),
            "supporting_evidence_summary": cap.get("supporting_evidence_summary", ""),
            "cautions": cap.get("cautions", []),
        }
        for cap in capability_review.get("capabilities", [])
    ]


def compact_evidence_for_positioning(evidence_items):
    return [
        {
            "evidence_id": item.get("evidence_id"),
            "selection_tier": item.get("selection_tier"),
            "source_label": item.get("source_label"),
            "evidence_text": item.get("evidence_text"),
            "primary_supporting_signals": item.get("primary_supporting_signals", []),
            "cautions": item.get("cautions", []),
        }
        for item in evidence_items
    ]


positioning_capabilities = compact_capabilities_for_positioning(capability_review)
positioning_evidence_compact = compact_evidence_for_positioning(positioning_evidence)

len(positioning_capabilities), len(positioning_evidence_compact)

(12, 23)

## 6C. Build Resume Positioning Prompt

In [6]:
def build_resume_positioning_prompt(
    positioning_capabilities,
    positioning_evidence,
    target_archetype,
):
    expected = {
        "positioning_summary": {
            "target_archetype": target_archetype.get("title", ""),
            "source_artifacts": [
                "capability_review.json",
                "selected_evidence.json",
                "target_archetype.json",
            ],
            "method": "Generate resume Summary and Core Expertise from evidence-backed capabilities."
        },
        "target_title": "string",
        "summary": {
            "text": "string",
            "supporting_capabilities": ["string"],
            "supporting_evidence_ids": ["string"],
            "cautions": ["string"]
        },
        "core_expertise": {
            "items": [
                {
                    "label": "string",
                    "category": "architecture | genai | governance | operations | leadership | data | business",
                    "use_priority": "primary | secondary",
                    "supporting_capabilities": ["string"],
                    "supporting_evidence_ids": ["string"]
                }
            ]
        },
        "positioning_notes": {
            "lead_story": "string",
            "avoid_overstating": ["string"],
            "experience_section_guidance": ["string"]
        }
    }

    return f"""
You are generating the top positioning section of a targeted resume.

This is resume positioning, not full resume assembly.

Target archetype:
{target_archetype.get("title", "")}

Archetype summary:
{target_archetype.get("archetype_summary", "")}

Evidence-backed capabilities:
{json.dumps(positioning_capabilities, indent=2)}

Selected evidence:
{json.dumps(positioning_evidence, indent=2)}

Task:
Generate:
1. A concise resume Summary.
2. A Core Expertise section.

Rules:
- Use only the provided capabilities and selected evidence.
- Do not introduce unsupported claims.
- Do not invent employers, technologies, metrics, responsibilities, or outcomes.
- Do not overstate teaching evidence as production ownership.
- Do not overstate roadmap, assessment, or prototype evidence as production implementation.
- Preserve important cautions in the output.
- Write in a polished but plain professional resume style.
- Avoid generic hype phrases like "visionary", "world-class", "rockstar", or "thought leader".
- The summary should be 3 to 4 sentences.
- The summary should emphasize current AI architecture, governed GenAI/tool-calling systems, observability, platforms, and technical leadership.
- Core Expertise should contain 10 to 14 items.
- Core Expertise labels should be concise, usually 2 to 5 words.
- Core Expertise labels should sound candidate-owned, not copied from job descriptions.
- Every Core Expertise item must have at least one supporting capability and one supporting evidence_id.
- Return valid JSON only.

Core Expertise categories:
- architecture
- genai
- governance
- operations
- leadership
- data
- business

Return exactly this structure:
{json.dumps(expected, indent=2)}
"""

## 6D. Generate Resume Positioning

In [7]:
prompt = build_resume_positioning_prompt(
    positioning_capabilities=positioning_capabilities,
    positioning_evidence=positioning_evidence_compact,
    target_archetype=target_archetype,
)

resume_positioning = call_json_model(prompt, MODEL)

resume_positioning.keys()

dict_keys(['positioning_summary', 'target_title', 'summary', 'core_expertise', 'positioning_notes'])

## 6E. Validate Resume Positioning

In [8]:
valid_evidence_ids = {
    item["evidence_id"]
    for item in positioning_evidence
}

valid_capabilities = {
    cap["capability"]
    for cap in positioning_capabilities
}

VALID_CATEGORIES = {
    "architecture",
    "genai",
    "governance",
    "operations",
    "leadership",
    "data",
    "business",
}

VALID_PRIORITIES = {
    "primary",
    "secondary",
}


def validate_resume_positioning(resume_positioning, valid_evidence_ids, valid_capabilities):
    errors = []

    summary = resume_positioning.get("summary", {})
    summary_text = summary.get("text", "")

    if not summary_text:
        errors.append("summary.text is missing")

    if len(summary_text.split(".")) < 3:
        errors.append("summary.text may be too short")

    for evidence_id in summary.get("supporting_evidence_ids", []):
        if evidence_id not in valid_evidence_ids:
            errors.append(f"summary has unknown evidence_id: {evidence_id}")

    for capability in summary.get("supporting_capabilities", []):
        if capability not in valid_capabilities:
            errors.append(f"summary has unknown capability: {capability}")

    core_items = resume_positioning.get("core_expertise", {}).get("items", [])

    if not (10 <= len(core_items) <= 14):
        errors.append(f"core_expertise should have 10 to 14 items, found {len(core_items)}")

    labels = []

    for idx, item in enumerate(core_items):
        label = item.get("label", "")
        labels.append(label)

        if not label:
            errors.append(f"core_expertise.items[{idx}] missing label")

        if item.get("category") not in VALID_CATEGORIES:
            errors.append(
                f"core_expertise.items[{idx}] invalid category: {item.get('category')}"
            )

        if item.get("use_priority") not in VALID_PRIORITIES:
            errors.append(
                f"core_expertise.items[{idx}] invalid use_priority: {item.get('use_priority')}"
            )

        if not item.get("supporting_capabilities"):
            errors.append(f"core_expertise.items[{idx}] has no supporting_capabilities")

        if not item.get("supporting_evidence_ids"):
            errors.append(f"core_expertise.items[{idx}] has no supporting_evidence_ids")

        for evidence_id in item.get("supporting_evidence_ids", []):
            if evidence_id not in valid_evidence_ids:
                errors.append(
                    f"core_expertise.items[{idx}] unknown evidence_id: {evidence_id}"
                )

        for capability in item.get("supporting_capabilities", []):
            if capability not in valid_capabilities:
                errors.append(
                    f"core_expertise.items[{idx}] unknown capability: {capability}"
                )

    duplicate_labels = {
        label
        for label in labels
        if labels.count(label) > 1
    }

    if duplicate_labels:
        errors.append(f"Duplicate Core Expertise labels: {sorted(duplicate_labels)}")

    return errors


positioning_errors = validate_resume_positioning(
    resume_positioning=resume_positioning,
    valid_evidence_ids=valid_evidence_ids,
    valid_capabilities=valid_capabilities,
)

positioning_errors

[]

## 6F. Review Summary and Core Expertise

In [9]:
print(resume_positioning["target_title"])

print("\nSUMMARY")
print(resume_positioning["summary"]["text"])

print("\nCORE EXPERTISE")
for item in resume_positioning["core_expertise"]["items"]:
    print(f"- {item['label']} [{item['category']}, {item['use_priority']}]")

Principal AI Architect / AI Platform Engineering Lead

SUMMARY
Senior technical leader with deep expertise architecting and delivering enterprise-scale AI/ML platforms, governed GenAI and tool-calling systems, and operational intelligence solutions. Proven track record in building production-ready analytics assistants with robust governance, observability, and operational controls. Experienced in cloud-native AI infrastructure, reusable frameworks, and cross-functional technical leadership, including mentoring and executive alignment. Adept at translating business needs into scalable, reliable AI solutions that drive measurable value.

CORE EXPERTISE
- Enterprise AI Platform Architecture [architecture, primary]
- Governed GenAI & Tool-Calling [genai, primary]
- AI Governance & Controls [governance, primary]
- Observability & Monitoring Frameworks [operations, primary]
- Cloud-Native AI Infrastructure [architecture, primary]
- Reusable Analytics Frameworks [data, secondary]
- Technical 

## Build Review Table

In [10]:
def build_core_expertise_review_table(resume_positioning):
    rows = []

    for item in resume_positioning.get("core_expertise", {}).get("items", []):
        rows.append({
            "label": item.get("label"),
            "category": item.get("category"),
            "use_priority": item.get("use_priority"),
            "capability_count": len(item.get("supporting_capabilities", [])),
            "evidence_count": len(item.get("supporting_evidence_ids", [])),
            "supporting_capabilities": "; ".join(item.get("supporting_capabilities", [])),
            "supporting_evidence_ids": ", ".join(item.get("supporting_evidence_ids", [])),
        })

    return pd.DataFrame(rows)


core_expertise_review_df = build_core_expertise_review_table(resume_positioning)
core_expertise_review_df

,label,category,use_priority,capability_count,evidence_count,supporting_capabilities,supporting_evidence_ids
0,Enterprise AI Platform Architecture,architecture,primary,1,3,Enterprise AI and Operational Intelligence Pla...,"exp_01_01_01_04, exp_01_01_02_01, exp_01_09_03_01"
1,Governed GenAI & Tool-Calling,genai,primary,1,2,Governed GenAI and Tool-Calling Systems,"exp_01_01_01_04, exp_01_02_02_01"
2,AI Governance & Controls,governance,primary,1,3,"AI Governance, Controls, and Responsible Adoption","exp_01_01_01_04, exp_01_02_04_01, exp_01_02_02_01"
3,Observability & Monitoring Frameworks,operations,primary,1,3,"Observability, Monitoring, and Operational Con...","exp_01_05_04_01, exp_01_09_03_01, exp_01_01_01_04"
4,Cloud-Native AI Infrastructure,architecture,primary,1,3,Cloud-Native AI Infrastructure and Data Platforms,"exp_01_01_02_01, exp_01_09_04_01, exp_01_08_01_01"
5,Reusable Analytics Frameworks,data,secondary,1,2,Reusable Frameworks and Developer Enablement,"exp_01_05_04_01, exp_01_01_01_02"
6,Technical Leadership & Mentorship,leadership,primary,1,3,"Technical Leadership, Mentorship, and Talent D...","exp_01_10_01_01, exp_01_02_04_01, exp_01_03_03_01"
7,Cross-Functional Executive Alignment,leadership,secondary,1,3,Cross-Functional Strategy and Executive Alignment,"exp_01_10_02_01, exp_01_09_05_01, exp_01_04_04_01"
8,Business-Driven AI Solutions,business,secondary,1,3,Business Translation and Decision Support,"exp_01_09_03_01, exp_01_08_01_01, exp_01_04_04_01"
9,AI/ML Model Lifecycle Management,architecture,secondary,1,1,AI/ML Deployment and Lifecycle Support,exp_01_01_02_01


## 6H. Save

In [11]:
save_json(resume_positioning, ARTIFACT_DIR / "resume_positioning.json")

core_expertise_review_df.to_csv(
    ARTIFACT_DIR / "resume_positioning_core_expertise_review.csv",
    index=False,
)

### Possible core expertise section
CORE EXPERTISE

AI Platform Architecture - enterprise AI platforms, governed analytics assistants, tool-calling systems, operational intelligence platforms, reusable AI/analytics frameworks

GenAI & Agentic Systems - LLM applications, structured outputs, tool orchestration, RAG/GraphRAG concepts, prompt engineering, evaluation, traceability, Gemini/OpenAI, LangChain

AI Governance & Controls - responsible AI practices, guardrails, auditable execution, analyst-in-the-loop workflows, policy-aware system design, risk-aware adoption

MLOps & Observability - monitoring frameworks, incident detection, alerting, operational health, SLIs/SLOs, SEV analysis, release hygiene, on-call-friendly design

Cloud & Data Platforms - Python, SQL, BigQuery, AWS SageMaker/S3/Terraform, GCP, Docker, batch pipelines, data profiling, validation, cross-system integration

Technical Leadership - architecture direction, executive alignment, technical roadmaps, stakeholder translation, mentoring, developer enablement, cross-functional delivery

## 6I. Convert Capability Items into Resume-Ready Core Expertise

The capability review produces evidence-backed capability themes. This section converts those themes into a compact Core Expertise block suitable for a resume.

This is a presentation step. The grouped lines may combine multiple evidence-backed capabilities, but they should not introduce unsupported technologies or claims.

Suggested output shape:
``` json
{
  "core_expertise_resume_block": {
    "heading": "CORE EXPERTISE",
    "style": "grouped_skill_lines",
    "items": [
      {
        "label": "AI Platform Architecture",
        "text": "enterprise AI platforms, governed analytics assistants, tool-calling systems, operational intelligence platforms, reusable AI/analytics frameworks",
        "supporting_capabilities": [
          "Enterprise AI Platform Architecture",
          "Production Analytics Assistants",
          "Operational Intelligence Platforms"
        ],
        "supporting_evidence_ids": ["..."]
      }
    ]
  }
}
```

In [12]:
def build_core_expertise_resume_block_prompt(resume_positioning):
    expected = {
        "core_expertise_resume_block": {
            "heading": "CORE EXPERTISE",
            "style": "grouped_skill_lines",
            "items": [
                {
                    "label": "string",
                    "text": "string",
                    "supporting_capabilities": ["string"],
                    "supporting_evidence_ids": ["string"]
                }
            ]
        }
    }

    return f"""
You are converting evidence-backed Core Expertise capability items into a resume-ready Core Expertise section.

Input Core Expertise capability items:
{json.dumps(resume_positioning["core_expertise"]["items"], indent=2)}

Task:
Create a compact resume-ready Core Expertise block.

Rules:
- Group related capabilities into 5 to 7 resume lines.
- Each line should have a concise label and a comma-separated skill/theme description.
- Do not introduce unsupported tools, technologies, or claims.
- It is acceptable to combine multiple capability items into one resume line.
- Avoid generic filler.
- Use a style similar to:
  "AI Platform Architecture - enterprise AI platforms, governed analytics assistants, tool-calling systems"
- Preserve traceability by listing supporting_capabilities and supporting_evidence_ids.
- Return valid JSON only.

Return exactly this structure:
{json.dumps(expected, indent=2)}
"""

In [13]:
prompt = build_core_expertise_resume_block_prompt(resume_positioning)
core_expertise_resume_block = call_json_model(prompt, MODEL)

resume_positioning["core_expertise_resume_block"] = core_expertise_resume_block[
    "core_expertise_resume_block"
]

save_json(resume_positioning, ARTIFACT_DIR / "resume_positioning.json")

## 6J. Human-Readable Resume Positioning Review

Display the generated Summary and Core Expertise sections in a resume-like format so they can be reviewed before full resume assembly.

In [15]:
def display_resume_positioning(resume_positioning):
    """
    Display resume positioning in a human-readable resume-like format.

    This is a review helper only. It does not modify the artifact.
    """
    print("=" * 100)
    print("TARGET TITLE")
    print("=" * 100)
    print(resume_positioning.get("target_title", ""))

    print("\n" + "=" * 100)
    print("SUMMARY")
    print("=" * 100)
    print(resume_positioning.get("summary", {}).get("text", ""))

    print("\n" + "=" * 100)
    print("CORE EXPERTISE")
    print("=" * 100)

    block = resume_positioning.get("core_expertise_resume_block", {})
    items = block.get("items", [])

    if items:
        for item in items:
            label = item.get("label", "")
            text = item.get("text", "")
            print(f"{label} - {text}")
    else:
        # Fallback if grouped resume block has not been generated yet.
        for item in resume_positioning.get("core_expertise", {}).get("items", []):
            label = item.get("label", "")
            category = item.get("category", "")
            priority = item.get("use_priority", "")
            print(f"- {label} [{category}, {priority}]")

    print("\n" + "=" * 100)
    print("POSITIONING NOTES")
    print("=" * 100)

    notes = resume_positioning.get("positioning_notes", {})

    lead_story = notes.get("lead_story")
    if lead_story:
        print("\nLead story:")
        print(lead_story)

    avoid_overstating = notes.get("avoid_overstating", [])
    if avoid_overstating:
        print("\nAvoid overstating:")
        for item in avoid_overstating:
            print(f"- {item}")

    guidance = notes.get("experience_section_guidance", [])
    if guidance:
        print("\nExperience section guidance:")
        for item in guidance:
            print(f"- {item}")

In [16]:
display_resume_positioning(resume_positioning)

TARGET TITLE
Principal AI Architect / AI Platform Engineering Lead

SUMMARY
Senior technical leader with deep expertise architecting and delivering enterprise-scale AI/ML platforms, governed GenAI and tool-calling systems, and operational intelligence solutions. Proven track record in building production-ready analytics assistants with robust governance, observability, and operational controls. Experienced in cloud-native AI infrastructure, reusable frameworks, and cross-functional technical leadership, including mentoring and executive alignment. Adept at translating business needs into scalable, reliable AI solutions that drive measurable value.

CORE EXPERTISE
Enterprise AI Platform Architecture - enterprise AI platforms, operational intelligence, cloud-native AI infrastructure, model lifecycle management
Governed GenAI & Analytics Assistants - governed GenAI, tool-calling systems, production analytics assistants
AI Governance & Controls - AI governance, operational controls, respon

In [17]:
def build_core_expertise_resume_block_df(resume_positioning):
    block = resume_positioning.get("core_expertise_resume_block", {})

    rows = []
    for item in block.get("items", []):
        rows.append({
            "label": item.get("label", ""),
            "text": item.get("text", ""),
            "supporting_capability_count": len(item.get("supporting_capabilities", [])),
            "supporting_evidence_count": len(item.get("supporting_evidence_ids", [])),
            "supporting_capabilities": "; ".join(item.get("supporting_capabilities", [])),
            "supporting_evidence_ids": ", ".join(item.get("supporting_evidence_ids", [])),
        })

    return pd.DataFrame(rows)


core_expertise_block_df = build_core_expertise_resume_block_df(resume_positioning)
core_expertise_block_df

,label,text,supporting_capability_count,supporting_evidence_count,supporting_capabilities,supporting_evidence_ids
0,Enterprise AI Platform Architecture,"enterprise AI platforms, operational intellige...",3,5,Enterprise AI and Operational Intelligence Pla...,"exp_01_01_01_04, exp_01_01_02_01, exp_01_09_03..."
1,Governed GenAI & Analytics Assistants,"governed GenAI, tool-calling systems, producti...",1,2,Governed GenAI and Tool-Calling Systems,"exp_01_01_01_04, exp_01_02_02_01"
2,AI Governance & Controls,"AI governance, operational controls, responsib...",2,4,"AI Governance, Controls, and Responsible Adopt...","exp_01_01_01_04, exp_01_02_04_01, exp_01_02_02..."
3,Observability & Monitoring Frameworks,"observability frameworks, monitoring, operatio...",2,3,"Observability, Monitoring, and Operational Con...","exp_01_05_04_01, exp_01_09_03_01, exp_01_01_01_04"
4,Advanced & Reusable Analytics,"advanced analytics, decision support, reusable...",3,6,Advanced Analytics and Decision Support; Reusa...,"exp_01_07_01_01, exp_01_05_04_01, exp_01_01_01..."
5,Technical Leadership & Strategy,"technical leadership, mentorship, executive al...",3,7,"Technical Leadership, Mentorship, and Talent D...","exp_01_10_01_01, exp_01_02_04_01, exp_01_03_03..."


In [18]:
core_expertise_block_df.to_csv(
    ARTIFACT_DIR / "resume_positioning_core_expertise_block_review.csv",
    index=False,
)

In [19]:
MANUAL_POSITIONING_PATCH = {
    "summary_text": """Senior technical leader with deep expertise architecting enterprise AI/ML platforms, governed GenAI and tool-calling systems, and operational intelligence solutions. Built governed analytics assistants, monitoring frameworks, and reusable data/AI platforms with strong emphasis on controls, observability, and auditable execution. Experienced across cloud-native AI infrastructure, large-scale data platforms, executive alignment, and technical mentorship. Known for translating ambiguous business needs into scalable, reliable analytical systems that improve decision-making and operational performance.""",
    "core_expertise_resume_block": {
        "heading": "CORE EXPERTISE",
        "style": "grouped_skill_lines",
        "items": [
            {
                "label": "AI Platform Architecture",
                "text": "enterprise AI platforms, governed analytics assistants, operational intelligence systems, reusable AI/analytics frameworks",
            },
            {
                "label": "GenAI & Tool-Calling Systems",
                "text": "LLM applications, structured outputs, tool orchestration, RAG/GraphRAG concepts, prompt engineering, evaluation, traceability",
            },
            {
                "label": "Governance & Operational Controls",
                "text": "responsible AI adoption, guardrails, auditable execution, analyst-in-the-loop workflows, policy-aware system design",
            },
            {
                "label": "Observability & Reliability",
                "text": "monitoring frameworks, incident detection, alerting, operational health, SLIs/SLOs, SEV analysis, on-call-friendly design",
            },
            {
                "label": "Cloud & Data Platforms",
                "text": "Python, SQL, BigQuery, AWS SageMaker/S3/Terraform, GCP, Docker, data profiling, validation, cross-system integration",
            },
            {
                "label": "Technical Leadership",
                "text": "architecture direction, executive alignment, technical roadmaps, stakeholder translation, mentoring, developer enablement",
            },
        ],
    },
}

In [20]:
resume_positioning["summary"]["original_text"] = resume_positioning["summary"]["text"]
resume_positioning["summary"]["text"] = MANUAL_POSITIONING_PATCH["summary_text"]

resume_positioning["core_expertise_resume_block_original"] = resume_positioning.get(
    "core_expertise_resume_block"
)

resume_positioning["core_expertise_resume_block"] = MANUAL_POSITIONING_PATCH[
    "core_expertise_resume_block"
]

save_json(resume_positioning, ARTIFACT_DIR / "resume_positioning.json")

## 6K. Apply Manual Resume Positioning Patch with Traceability

The grouped Core Expertise block is manually polished for resume readability. This section reattaches supporting capabilities and evidence IDs from the generated Core Expertise table so the final artifact remains traceable.

In [21]:
MANUAL_CORE_EXPERTISE_PATCH = [
    {
        "label": "AI Platform Architecture",
        "text": "enterprise AI platforms, governed analytics assistants, operational intelligence systems, reusable AI/analytics frameworks",
        "source_core_expertise_labels": [
            "Enterprise AI Platform Architecture",
            "Cloud-Native AI Infrastructure",
            "Operational Intelligence Platforms",
            "Reusable Analytics Frameworks",
            "AI/ML Model Lifecycle Management",
        ],
    },
    {
        "label": "GenAI & Tool-Calling Systems",
        "text": "LLM applications, structured outputs, tool orchestration, RAG/GraphRAG concepts, prompt engineering, evaluation, traceability",
        "source_core_expertise_labels": [
            "Governed GenAI & Tool-Calling",
            "Production Analytics Assistants",
        ],
    },
    {
        "label": "Governance & Operational Controls",
        "text": "responsible AI adoption, guardrails, auditable execution, analyst-in-the-loop workflows, policy-aware system design",
        "source_core_expertise_labels": [
            "AI Governance & Controls",
            "Governed GenAI & Tool-Calling",
        ],
    },
    {
        "label": "Observability & Reliability",
        "text": "monitoring frameworks, incident detection, alerting, operational health, SLIs/SLOs, SEV analysis, on-call-friendly design",
        "source_core_expertise_labels": [
            "Observability & Monitoring Frameworks",
            "Operational Intelligence Platforms",
        ],
    },
    {
        "label": "Cloud & Data Platforms",
        "text": "Python, SQL, BigQuery, AWS SageMaker/S3/Terraform, GCP, Docker, data profiling, validation, cross-system integration",
        "source_core_expertise_labels": [
            "Cloud-Native AI Infrastructure",
            "Reusable Analytics Frameworks",
            "Advanced Analytics & Decision Support",
        ],
    },
    {
        "label": "Technical Leadership",
        "text": "architecture direction, executive alignment, technical roadmaps, stakeholder translation, mentoring, developer enablement",
        "source_core_expertise_labels": [
            "Technical Leadership & Mentorship",
            "Cross-Functional Executive Alignment",
            "Strategy & Technical Roadmaps",
            "Business-Driven AI Solutions",
        ],
    },
]

In [22]:
def apply_manual_core_expertise_patch_with_traceability(
    resume_positioning,
    manual_patch,
):
    """
    HUMAN_REVIEW PATCH.

    Applies a resume-ready Core Expertise block while preserving traceability
    to the generated Core Expertise items, supporting capabilities, and evidence IDs.

    This is not a STUB. The generated table provides evidence-based structure;
    this manual patch provides candidate-specific resume wording.
    """
    generated_items = resume_positioning.get("core_expertise", {}).get("items", [])

    generated_by_label = {
        item.get("label"): item
        for item in generated_items
    }

    patched_items = []

    for patched in manual_patch:
        source_labels = patched.get("source_core_expertise_labels", [])

        missing_labels = [
            label
            for label in source_labels
            if label not in generated_by_label
        ]

        supporting_capabilities = []
        supporting_evidence_ids = []
        supporting_categories = []
        supporting_priorities = []

        for label in source_labels:
            source_item = generated_by_label.get(label)

            if source_item is None:
                continue

            supporting_capabilities.extend(
                source_item.get("supporting_capabilities", [])
            )
            supporting_evidence_ids.extend(
                source_item.get("supporting_evidence_ids", [])
            )

            if source_item.get("category"):
                supporting_categories.append(source_item["category"])

            if source_item.get("use_priority"):
                supporting_priorities.append(source_item["use_priority"])

        patched_items.append({
            "label": patched["label"],
            "text": patched["text"],
            "source_core_expertise_labels": source_labels,
            "missing_source_core_expertise_labels": missing_labels,
            "supporting_capabilities": sorted(set(supporting_capabilities)),
            "supporting_evidence_ids": sorted(set(supporting_evidence_ids)),
            "supporting_categories": sorted(set(supporting_categories)),
            "supporting_priorities": sorted(set(supporting_priorities)),
        })

    resume_positioning["core_expertise_resume_block_original"] = resume_positioning.get(
        "core_expertise_resume_block"
    )

    resume_positioning["core_expertise_resume_block"] = {
        "heading": "CORE EXPERTISE",
        "style": "grouped_skill_lines",
        "items": patched_items,
        "patch_method": (
            "Manual resume wording mapped back to generated Core Expertise items "
            "to preserve capability and evidence traceability."
        ),
    }

    return resume_positioning


resume_positioning = apply_manual_core_expertise_patch_with_traceability(
    resume_positioning=resume_positioning,
    manual_patch=MANUAL_CORE_EXPERTISE_PATCH,
)

In [23]:
MANUAL_SUMMARY_TEXT = (
    "Senior technical leader with deep expertise architecting enterprise AI/ML platforms, "
    "governed GenAI and tool-calling systems, and operational intelligence solutions. "
    "Built governed analytics assistants, monitoring frameworks, and reusable data/AI "
    "platforms with strong emphasis on controls, observability, and auditable execution. "
    "Experienced across cloud-native AI infrastructure, large-scale data platforms, "
    "executive alignment, and technical mentorship. Known for translating ambiguous "
    "business needs into scalable, reliable analytical systems that improve "
    "decision-making and operational performance."
)

resume_positioning["summary"]["original_text"] = resume_positioning["summary"]["text"]
resume_positioning["summary"]["text"] = MANUAL_SUMMARY_TEXT
resume_positioning["summary"]["patch_method"] = (
    "Manual wording pass to improve precision and avoid overstating production readiness."
)

In [24]:
def validate_patched_core_expertise_block(resume_positioning):
    errors = []

    block = resume_positioning.get("core_expertise_resume_block", {})
    items = block.get("items", [])

    if not items:
        errors.append("core_expertise_resume_block has no items")

    for idx, item in enumerate(items):
        if not item.get("label"):
            errors.append(f"items[{idx}] missing label")

        if not item.get("text"):
            errors.append(f"items[{idx}] missing text")

        if not item.get("supporting_capabilities"):
            errors.append(f"items[{idx}] has no supporting_capabilities")

        if not item.get("supporting_evidence_ids"):
            errors.append(f"items[{idx}] has no supporting_evidence_ids")

        if item.get("missing_source_core_expertise_labels"):
            errors.append(
                f"items[{idx}] has missing source labels: "
                f"{item['missing_source_core_expertise_labels']}"
            )

    return errors


patch_errors = validate_patched_core_expertise_block(resume_positioning)
patch_errors

[]

In [25]:
save_json(resume_positioning, ARTIFACT_DIR / "resume_positioning.json")

In [28]:
resume_positioning.keys()

dict_keys(['positioning_summary', 'target_title', 'summary', 'core_expertise', 'positioning_notes', 'core_expertise_resume_block', 'core_expertise_resume_block_original'])

In [26]:
display_resume_positioning(resume_positioning)

TARGET TITLE
Principal AI Architect / AI Platform Engineering Lead

SUMMARY
Senior technical leader with deep expertise architecting enterprise AI/ML platforms, governed GenAI and tool-calling systems, and operational intelligence solutions. Built governed analytics assistants, monitoring frameworks, and reusable data/AI platforms with strong emphasis on controls, observability, and auditable execution. Experienced across cloud-native AI infrastructure, large-scale data platforms, executive alignment, and technical mentorship. Known for translating ambiguous business needs into scalable, reliable analytical systems that improve decision-making and operational performance.

CORE EXPERTISE
AI Platform Architecture - enterprise AI platforms, governed analytics assistants, operational intelligence systems, reusable AI/analytics frameworks
GenAI & Tool-Calling Systems - LLM applications, structured outputs, tool orchestration, RAG/GraphRAG concepts, prompt engineering, evaluation, traceabil

## Stage 6 Completion Summary

Stage 6 converts evidence-backed capabilities into resume-ready positioning.

The main output is `resume_positioning.json`, which contains:

* Target title
* Resume Summary
* Traceable Core Expertise capability items
* Resume-ready grouped Core Expertise block
* Positioning notes and overstatement cautions

Key decisions:

* The Summary is generated from evidence-backed capabilities, then manually patched for precision and tone.
* The Core Expertise capability table is preserved for traceability.
* A separate resume-ready Core Expertise block is created for actual resume use.
* Manual wording patches are intentional human review steps, not model failures.
* Supporting capabilities and evidence IDs are reattached after manual edits so the final artifact remains auditable.

Stage 6 is complete when:

* `resume_positioning.json` is saved.
* The Summary reads like polished resume language.
* The Core Expertise block is compact and resume-ready.
* Each Core Expertise line maps back to supporting capabilities and evidence IDs.
* Overstatement risks are documented rather than hidden.
